# k-means++ and Large-Scale k-means

## Learning Objectives

1. **Recall** the Lloyd's algorithm and its convergence properties
2. **Implement** k-means++ initialization and explain its theoretical advantage
3. **Identify** the bottlenecks of k-means at scale
4. **Preview** BFR as the scalable extension

## Lloyd's Algorithm (Standard k-means)

Given $n$ points and $k$ clusters:

1. **Initialize** $k$ centroids $\mu_1, \ldots, \mu_k$ (randomly or by heuristic)
2. **Assign** each point to its nearest centroid:
   $z_i = \arg\min_j d(x_i, \mu_j)$
3. **Update** centroids: $\mu_j = \frac{1}{|C_j|} \sum_{i: z_i=j} x_i$
4. Repeat 2–3 until assignments don't change

**Convergence:** guaranteed (WCSS is non-increasing), but to a **local minimum**.

**Complexity per iteration:** $O(nkd)$

**Issue:** bad initialization can lead to poor solutions.

## k-means++ Initialization

**Idea:** choose initial centroids spread out across the data.

1. Choose $c_1$ uniformly at random from $X$
2. For $l = 2, \ldots, k$:
   - For each point $x_i$, compute $D(x_i) = \min_{j < l} d(x_i, c_j)^2$
   - Choose $c_l \propto D(x_i)$ (points far from existing centroids are more likely)

**Guarantee:** Expected WCSS $\leq 8(\ln k + 2) \cdot \text{OPT}$ (Arthur & Vassilvitskii 2007).

This is an $O(\log k)$ approximation to the optimal clustering, provably better than random initialization.

In [ ]:
import math, random

def dist2(x, y):
    return sum((a-b)**2 for a,b in zip(x,y))

def dist(x, y):
    return math.sqrt(dist2(x,y))

def kmeans_pp_init(points, k, seed=0):
    """k-means++ initialization."""
    rng = random.Random(seed)
    centroids = [rng.choice(points)]
    for _ in range(k-1):
        D = [min(dist2(p, c) for c in centroids) for p in points]
        total = sum(D)
        probs = [d/total for d in D]
        # Weighted random choice
        r = rng.random()
        cumulative = 0.0
        chosen = points[-1]
        for p, prob in zip(points, probs):
            cumulative += prob
            if r <= cumulative:
                chosen = p; break
        centroids.append(chosen)
    return centroids

def assign(points, centroids):
    return [min(range(len(centroids)), key=lambda j: dist2(p, centroids[j]))
            for p in points]

def update_centroids(points, labels, k, d):
    centroids = [[0.0]*d for _ in range(k)]
    counts = [0]*k
    for p, l in zip(points, labels):
        for i in range(d): centroids[l][i] += p[i]
        counts[l] += 1
    return [[c[i]/counts[j] if counts[j] > 0 else 0.0 for i in range(d)]
            for j,c in enumerate(centroids)]

def wcss(points, labels, centroids):
    return sum(dist2(p, centroids[l]) for p,l in zip(points,labels))

def kmeans(points, k, init='pp', max_iter=100, seed=0):
    d = len(points[0])
    rng = random.Random(seed)
    if init == 'pp':
        centroids = kmeans_pp_init(points, k, seed)
    else:
        centroids = rng.sample(points, k)
    for _ in range(max_iter):
        labels = assign(points, centroids)
        new_c = update_centroids(points, labels, k, d)
        if all(dist2(a,b) < 1e-8 for a,b in zip(centroids, new_c)):
            break
        centroids = new_c
    return labels, centroids, wcss(points, labels, centroids)

# Generate data
rng = random.Random(1)
k_true = 4
points = []
centers_true = [(0,0),(10,0),(5,8),(0,10)]
for cx,cy in centers_true:
    for _ in range(50):
        points.append([cx+rng.gauss(0,1.5), cy+rng.gauss(0,1.5)])

# Compare random vs k-means++ over multiple trials
print("WCSS comparison (lower is better):")
print(f"{'Trial':>6} {'Random init':>14} {'k-means++':>14}")
wcss_random, wcss_pp = [], []
for t in range(20):
    _, _, w_rand = kmeans(points, 4, init='random', seed=t*17)
    _, _, w_pp   = kmeans(points, 4, init='pp',     seed=t*17)
    wcss_random.append(w_rand); wcss_pp.append(w_pp)

print(f"Mean:   {sum(wcss_random)/len(wcss_random):>14.0f} {sum(wcss_pp)/len(wcss_pp):>14.0f}")
print(f"Min:    {min(wcss_random):>14.0f} {min(wcss_pp):>14.0f}")
print(f"Max:    {max(wcss_random):>14.0f} {max(wcss_pp):>14.0f}")

## Scalability Limitations

| Constraint | k-means | Large-scale need |
|-----------|---------|-----------------|
| Data in memory | All $n$ points | Disk-resident; load in chunks |
| Passes | Multiple per iteration | 1–2 total |
| Outliers | Distort centroids | Need robustness |
| Cluster shape | Spherical (Voronoi) | Non-spherical clusters |

**BFR (Bradley-Fayyad-Reina)** extends k-means to disk-resident data by:
- Processing data in chunks (load one chunk at a time)
- Maintaining sufficient statistics $(N, SUM, SUMSQ)$ per cluster
- "Compressing" nearby non-centroid points into sub-clusters
- "Retaining" true outliers in a separate set

BFR makes k-means practical for datasets too large to fit in RAM.